8 de mayo 2026

Regina Tamayo León


# **T10 - Particiones en árboles de clasificación**

***Conceptos y su conexión con las particiones en árboles de clasificación:***

 __________________
**GINI:**

Es el criterio que utiliza el algoritmo CART (Classification and Regression Trees). Mide qué tan seguido se elegiría incorrectamente un elemento del conjunto si se etiquetara aleatoriamente de acuerdo con la distribución de clases en el nodo.

*Conexión con las particiones:*

El árbol calcula el Gini de cada nodo hijo después de un posible split. El objetivo es minimizar la impureza. Si un nodo es "puro" (todos son de la misma clase), el Gini es 0.

Cálculo: Se basa en la suma de los cuadrados de las probabilidades de cada clase:$$G = 1 - \sum_{i=1}^{J} p_i^2$$

**Entropía:**

Proveniente de la teoría de la información, mide el nivel de "desorden" o incertidumbre en los datos. Es el criterio principal en algoritmos como ID3 o C4.5.

*Conexión con las particiones:*

Se utiliza para calcular la Ganancia de Información (Information Gain). El árbol elige el split que genera la mayor reducción de entropía entre el nodo padre y los nodos hijos.

Cálculo: Utiliza logaritmos, lo que la hace computacionalmente un poco más pesada que Gini, pero más sensible a cambios en las probabilidades:$$H = -\sum_{i=1}^{J} p_i \log_2(p_i)$$

**Log Loss:**

Aunque matemáticamente es casi idéntica a la entropía, el Log Loss se entiende más como una función de costo que penaliza las predicciones erróneas, especialmente aquellas en las que el modelo estaba muy "seguro".

*Conexión con las particiones:*

En árboles de decisión simples no es el criterio estándar para partir (se prefieren Gini o Entropía). Sin embargo, es fundamental en modelos de Gradient Boosting. En estos casos, los árboles sucesivos se construyen para minimizar el Log Loss del modelo global.

 __________________

**¿Cuál es la diferencia entre entropía y log loss?:**

1. El Objeto de Medida:

La Entropía mide la incertidumbre de una variable aleatoria dentro de un solo conjunto (el nodo).

El Log Loss mide la distancia entre dos distribuciones: la probabilidad predicha por el modelo y el valor real (0 o 1).

2. El Momento de Uso:

Usas Entropía para decidir cómo dividir los datos en la fase de entrenamiento del árbol.

Usas Log Loss para evaluar qué tan bueno es tu clasificador al asignar probabilidades a sus predicciones.

3. Escala:

La entropía suele calcularse con $\log_2$ (midiendo la información en "bits").

El Log Loss suele usar el logaritmo natural ($\ln$), ya que facilita el cálculo de derivadas en algoritmos de optimización.

In [25]:
import pandas as pd
import numpy as np

df = pd.read_csv('brain_tumor_dataset.csv')
df.head()

,Patient_ID,Age,Gender,Tumor_Type,Tumor_Size,Location,Histology,Stage,Symptom_1,Symptom_2,Symptom_3,Radiation_Treatment,Surgery_Performed,Chemotherapy,Survival_Rate,Tumor_Growth_Rate,Family_History,MRI_Result,Follow_Up_Required
0,1,73,Male,Malignant,5.375612,Temporal,Astrocytoma,III,Vision Issues,Seizures,Seizures,No,No,No,51.312579,0.111876,No,Positive,Yes
1,2,26,Male,Benign,4.847098,Parietal,Glioblastoma,II,Headache,Headache,Nausea,Yes,Yes,Yes,46.373273,2.165736,Yes,Positive,Yes
2,3,31,Male,Benign,5.588391,Parietal,Meningioma,I,Vision Issues,Headache,Seizures,No,No,No,47.072221,1.884228,No,Negative,No
3,4,29,Male,Malignant,1.436600,Temporal,Medulloblastoma,IV,Vision Issues,Seizures,Headache,Yes,No,Yes,51.853634,1.283342,Yes,Negative,No
4,5,54,Female,Benign,2.417506,Parietal,Glioblastoma,I,Headache,Headache,Seizures,No,No,Yes,54.708987,2.069477,No,Positive,Yes


Partición específica: ¿Qué tan bien separa los tumores benignos de los malignos el resultado de la resonancia (MRI_Result)?

In [33]:
# 0 = Benigno, 1 = Maligno
df['target'] = df['Tumor_Type'].map({'Benign': 0, 'Malignant': 1})

y_padre = df['y'] # Nodo Padre

y_izq = df[df['MRI_Result'] == 'Positive']['y']  
y_der = df[df['MRI_Result'] == 'Negative']['y']  

Criterio 1: Gini (Impureza)

In [27]:
def calcular_gini(y):
    if len(y) == 0: return 0
    p = y.value_counts(normalize=True).values
    return 1 - np.sum(p**2)


Criterio 2: Entropía (Incertidumbre)

In [28]:
def calcular_entropia(y):
    if len(y) == 0: return 0
    p = y.value_counts(normalize=True).values
    return -np.sum(p * np.log2(p + 1e-15))


Criterio 3: Log Loss (Pérdida Logarítmica)

In [29]:
def calcular_logloss(y):
    if len(y) == 0: return 0
    p_mean = y.mean()
    p_mean = np.clip(p_mean, 1e-15, 1 - 1e-15)
    return -np.mean(y * np.log(p_mean) + (1 - y) * np.log(1 - p_mean))


Split: dos grupos basados en el MRI

In [34]:
node_padre = df['target']
node_pos = df[df['MRI_Result'] == 'Positive']['target']
node_neg = df[df['MRI_Result'] == 'Negative']['target']

# Pesos (proporción de pacientes en cada grupo)
w_pos = len(node_pos) / len(node_padre)
w_neg = len(node_neg) / len(node_padre)


Calcular y comparar

In [35]:
metricas = [('Gini', calcular_gini), ('Entropía', calcular_entropia), ('Log Loss', calcular_logloss)]

print(f"{'Criterio':<12} | {'Padre':<10} | {'Hijos W':<10} | {'Ganancia':<10}")
print("-" * 55)

for nombre, func in metricas:
    imp_padre = func(node_padre)
    imp_hijos = (w_pos * func(node_pos)) + (w_neg * func(node_neg))
    ganancia = imp_padre - imp_hijos
    print(f"{nombre:<12} | {imp_padre:.6f} | {imp_hijos:.6f} | {ganancia:.8f}")


Criterio     | Padre      | Hijos W    | Ganancia  
-------------------------------------------------------
Gini         | 0.499996 | 0.499995 | 0.00000040
Entropía     | 0.999994 | 0.999993 | 0.00000057
Log Loss     | 0.693143 | 0.693142 | 0.00000040


In [36]:
print("ANÁLISIS DE DETECCIÓN")
print(f"Probabilidad de Maligno en MRI Positivo: {node_pos.mean():.4f}")
print(f"Probabilidad de Maligno en MRI Negativo: {node_neg.mean():.4f}")

ANÁLISIS DE DETECCIÓN
Probabilidad de Maligno en MRI Positivo: 0.5019
Probabilidad de Maligno en MRI Negativo: 0.5011


**Conclusiones**

A. El Nodo Padre (El desorden total)

dice que la impureza del Padre es casi 0.5 (Gini) o 1.0 (Entropía). Esto significa que si se toma un paciente al azar, se tiene un 50% de probabilidad de equivocación. El desorden es máximo.

B. Los Hijos (El desorden después de la partición)

Dividimos a los pacientes en dos salas: Sala de "MRI Positivo" y Sala de "MRI Negativo".
Si el MRI fuera un gran detector, una sala debería estar llena de Malignos y la otra de Benignos. El desorden (impureza) de las salas sería cercano a 0. En el resultado, la impureza de los hijos es casi igual a la del padre.

C. La Ganancia 

La Ganancia es la resta: Impureza Padre - Impureza Hijos.

La ganancia es  0.00000040. Es un número insignificante.
Interpretación: Al separar por MRI, no ganamos nada de orden. El desorden sigue ahí.

- ¿Es el MRI un detector? 

NO es un detector. Significa que el resultado del MRI no cambia las sospechas sobre el tumor.

Si la Ganancia es minúscula: El árbol de decisión dirá: "No voy a usar el MRI para mi primera rama"

Se ha demostrado matemáticamente que el MRI Result en este dataset es una variable irrelevante para clasificar, porque no genera una partición que limpie los datos.